In [ ]:
import pandas as pd
import openpyxl

df_district = pd.read_excel("../data/file/sdot_weather_data/서울시 도시데이터 센서(S-DoT) 환경정보 설치 위치정보.xlsx")
df_district["구"] = df_district["주소"].str.split(" ").str[1]


# df_district.sort_values("모델 시리얼(*)")
df_district.groupby("구").size()

df_gu = df_district.loc[:,["모델 시리얼(*)","구"]]
df_gu.head(2)

,모델 시리얼(*),구
0,V02Q1940655,종로구
1,V02Q1940539,종로구


In [ ]:
df_sample = pd.read_csv("../data/file/sdot_weather_data/S-DoT_NATURE_2020년(2020.04.01~2021.01.03)/S-DoT_NATURE_2020.04.01-04.30.csv", encoding = "utf-8").copy()

# df_sample.groupby("시리얼").size()
df_sample.nunique()

기관 명                1
모델명                 1
시리얼               857
구분                  1
 기온(℃)           1018
상대습도( %)           89
풍향(°)             340
풍속(m/s)            21
돌풍 풍향(°)          311
돌풍 풍속(m/s)        853
조도(lux)         60221
자외선(UVI)           93
소음(dB)             45
 진동_x(g)           15
 진동_y(g)          102
 진동_z(g)          831
 진동_x 최대(g)       969
 진동_y 최대(g)       398
 진동_z 최대(g)       741
흑구 온도(℃)        19736
전송시간              720
등록일자              720
dtype: int64

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

SDOT_DIR = Path("../data/file/sdot_weather_data")
DISTRICT_XLSX = SDOT_DIR / "서울시 도시데이터 센서(S-DoT) 환경정보 설치 위치정보.xlsx"

# 신규 포맷(2025~) 자치구 영문 → 한글 (xlsx 시리얼과 불일치 시 보조)
GU_EN_TO_KO = {
    "Jongno-gu": "종로구", "Jung-gu": "중구", "Yongsan-gu": "용산구",
    "Seongdong-gu": "성동구", "Gwangjin-gu": "광진구", "Dongdaemun-gu": "동대문구",
    "Jungnang-gu": "중랑구", "Seongbuk-gu": "성북구", "Gangbuk-gu": "강북구",
    "Dobong-gu": "도봉구", "Nowon-gu": "노원구", "Eunpyeong-gu": "은평구",
    "Seodaemun-gu": "서대문구", "Mapo-gu": "마포구", "Yangcheon-gu": "양천구",
    "Gangseo-gu": "강서구", "Guro-gu": "구로구", "Geumcheon-gu": "금천구",
    "Yeongdeungpo-gu": "영등포구", "Dongjak-gu": "동작구", "Gwanak-gu": "관악구",
    "Seocho-gu": "서초구", "Gangnam-gu": "강남구", "Songpa-gu": "송파구",
    "Gangdong-gu": "강동구",
}


def _clean_col(name: str) -> str:
    return re.sub(r"\s+", "", str(name).strip())


def load_district() -> pd.DataFrame:
    """센서 위치 xlsx → 시리얼·구 매핑."""
    df = pd.read_excel(DISTRICT_XLSX)
    df["구"] = df["주소"].str.split(" ").str[1]
    return df.rename(columns={"모델 시리얼(*)": "시리얼"})[["시리얼", "구", "주소"]]


def _read_sdot_csv(path: Path) -> pd.DataFrame:
    """연도별로 다른 S-DoT CSV 스키마를 통일 컬럼으로 읽기."""
    header = pd.read_csv(path, nrows=0, encoding="utf-8").columns.tolist()
    try:
        df = pd.read_csv(path, encoding="utf-8", usecols=header, low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="cp949", usecols=header, low_memory=False)

    df.columns = [_clean_col(c) for c in df.columns]

    rename = {
        "기온(℃)": "기온",
        "온도평균(℃)": "기온",
        "상대습도(%)": "상대습도",
        "습도평균(%)": "상대습도",
        "풍속(m/s)": "풍속",
        "풍속평균(m/s)": "풍속",
        "등록일시": "등록일자",
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    if "시리얼" not in df.columns:
        raise ValueError(f"시리얼 컬럼 없음: {path.name}")

    keep = [c for c in ["시리얼", "기온", "상대습도", "풍속", "등록일자", "자치구"] if c in df.columns]
    df = df[keep].copy()
    df["시리얼"] = df["시리얼"].astype(str).str.strip()
    df["등록일자"] = pd.to_datetime(df["등록일자"], errors="coerce")
    for col in ["기온", "상대습도", "풍속"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["year"] = df["등록일자"].dt.year
    return df


def load_sdot_weather(
    data_dir: Path | str = SDOT_DIR,
    years: list[int] | None = None,
    limit_files: int | None = None,
) -> pd.DataFrame:
    """sdot_weather_data 하위 모든 CSV 로드 후 통합."""
    data_dir = Path(data_dir)
    paths = sorted(data_dir.rglob("*.csv"))
    if limit_files:
        paths = paths[:limit_files]

    frames: list[pd.DataFrame] = []
    for path in paths:
        df = _read_sdot_csv(path)
        if years is not None:
            df = df[df["year"].isin(years)]
        if not df.empty:
            frames.append(df)

    if not frames:
        return pd.DataFrame(columns=["시리얼", "기온", "상대습도", "풍속", "등록일자", "year", "구"])

    return pd.concat(frames, ignore_index=True)


def merge_sdot_with_district(
    weather: pd.DataFrame,
    district: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """시리얼 = 모델 시리얼(*) 기준 left merge, 신규 데이터는 자치구 보조."""
    district = district if district is not None else load_district()
    district = district.copy()
    district["시리얼"] = district["시리얼"].astype(str).str.strip()

    merged = weather.merge(district[["시리얼", "구"]], on="시리얼", how="left")

    if "자치구" in merged.columns:
        fallback = merged["자치구"].map(GU_EN_TO_KO)
        merged["구"] = merged["구"].fillna(fallback)

    return merged


def to_year_json(
    df: pd.DataFrame,
    columns: list[str] | None = None,
) -> dict[str, list[dict]]:
    """연도별 JSON: {"2026": [{기온, 상대습도, ...}, ...], "2025": [...]}"""
    columns = columns or ["시리얼", "구", "기온", "상대습도", "풍속", "등록일자"]
    out: dict[str, list[dict]] = {}

    for year, group in df.dropna(subset=["year"]).groupby("year"):
        chunk = group[columns].copy()
        chunk["등록일자"] = chunk["등록일자"].dt.strftime("%Y-%m-%d %H:%M:%S")
        out[str(int(year))] = chunk.to_dict(orient="records")

    return dict(sorted(out.items(), key=lambda x: x[0], reverse=True))

In [19]:
# --- 데모: 소량 로드 → 머지 → 연도별 JSON ---
district = load_district()

# 전체 425개 CSV는 수백만 행이라 EDA는 연도·파일 수를 제한하는 것을 권장
weather = load_sdot_weather(years=[2020], limit_files=3)
df_merged = merge_sdot_with_district(weather, district)

print(df_merged.shape)
print(df_merged[["시리얼", "구", "기온", "상대습도", "풍속", "등록일자"]].head(3))

sdot_json = to_year_json(df_merged)
print("years:", list(sdot_json.keys()))
print("sample:", json.dumps(sdot_json["2020"][:2], ensure_ascii=False, indent=2))

# 파일로 저장 시 (용량 주의)
# with open("../data/file/sdot_year.json", "w", encoding="utf-8") as f:
#     json.dump(sdot_json, f, ensure_ascii=False)

(1636089, 7)
           시리얼    구    기온  상대습도   풍속                등록일자
0  V02Q1940043  구로구  14.0  37.0  0.0 2020-04-01 01:00:00
1  V02Q1940043  구로구  16.6  28.0  0.0 2020-04-01 12:00:00
2  V02Q1940043  구로구  15.0  16.0  0.0 2020-04-01 18:00:00
years: ['2020']
sample: [
  {
    "시리얼": "V02Q1940043",
    "구": "구로구",
    "기온": 14.0,
    "상대습도": 37.0,
    "풍속": 0.0,
    "등록일자": "2020-04-01 01:00:00"
  },
  {
    "시리얼": "V02Q1940043",
    "구": "구로구",
    "기온": 16.6,
    "상대습도": 28.0,
    "풍속": 0.0,
    "등록일자": "2020-04-01 12:00:00"
  }
]


### Q. JSON 형식이어도 pandas / GeoJSON 작업에 무리 없나?

**pandas** — 문제 없음. `pd.DataFrame(sdot_json["2026"])` 또는 `pd.json_normalize(sdot_json)` 으로 바로 표로 복원 가능. 분석·집계는 DataFrame으로 두고, API·저장용으로만 JSON을 쓰는 패턴이 일반적.

**GeoJSON** — 시계열 JSON과는 목적이 다름. GeoJSON은 `Point`/`Feature` + `geometry`(위도·경도)가 핵심. 구·기온 시계열을 지도에 올리려면 xlsx의 **위도·경도**를 머지한 뒤 `geopandas.GeoDataFrame` → `.to_file("sdot.geojson")` 형태가 맞음. 연도별 중첩 JSON을 GeoJSON으로 바로 쓰기보다, **분석용 DataFrame ↔ 배포용 JSON/GeoJSON** 으로 나누는 것을 권장.

**참고** — 2020~ 포맷은 `시리얼`(V02Q…)이 xlsx와 일치. 2025~ 신규 포맷은 `OC3CL…` 시리얼이라 xlsx 머지가 안 되므로, 위 코드는 `자치구` 영문→한글 매핑으로 `구`를 보완함.